In [1]:
# ============================================================
# CELL 1: Mount Drive & Cài thư viện
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

!pip install -q datasets transformers scikit-learn tqdm

Mounted at /content/drive


In [2]:
# ============================================================
# CELL 2: Import
# ============================================================
import os, json, warnings
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertModel
from datasets import load_dataset
from PIL import Image
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm import tqdm
from datetime import datetime

warnings.filterwarnings("ignore")

DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SAVE_DIR   = "/content/drive/MyDrive/BERT_MiRAGe"
BATCH_SIZE = 16
EPOCHS     = 5
LR         = 2e-5
MAX_LEN    = 256

os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Device  : {DEVICE}")
print(f"Save dir: {SAVE_DIR}")

Device  : cuda
Save dir: /content/drive/MyDrive/BERT_MiRAGe


In [3]:
# ============================================================
# CELL 3: Load dataset
# ============================================================
print("Đang tải dataset...")
raw = load_dataset("anson-huang/mirage-news")
print(raw)
print("\nFeatures:", raw["train"].features)
print("Mẫu đầu :", {k: str(v)[:80] for k, v in raw["train"][0].items()})

Đang tải dataset...


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/655M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/143M [00:00<?, ?B/s]

data/test1_nyt_mj-00000-of-00001.parquet:   0%|          | 0.00/20.2M [00:00<?, ?B/s]

...

Generating test2_bbc_dalle split:   0%|          | 0/500 [00:00<?, ? examples/s]

Generating test3_cnn_dalle split:   0%|          | 0/500 [00:00<?, ? examples/s]

Generating test4_bbc_sdxl split:   0%|          | 0/500 [00:00<?, ? examples/s]

Generating test5_cnn_sdxl split:   0%|          | 0/500 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['image', 'label', 'text'],
        num_rows: 10000
    })
    validation: Dataset({
        features: ['image', 'label', 'text'],
        num_rows: 2500
    })
    test1_nyt_mj: Dataset({
        features: ['image', 'label', 'text'],
        num_rows: 500
    })
    test2_bbc_dalle: Dataset({
        features: ['image', 'label', 'text'],
        num_rows: 500
    })
    test3_cnn_dalle: Dataset({
        features: ['image', 'label', 'text'],
        num_rows: 500
    })
    test4_bbc_sdxl: Dataset({
        features: ['image', 'label', 'text'],
        num_rows: 500
    })
    test5_cnn_sdxl: Dataset({
        features: ['image', 'label', 'text'],
        num_rows: 500
    })
})

Features: {'image': Image(mode=None, decode=True), 'label': ClassLabel(names=['real', 'fake']), 'text': Value('string')}
Mẫu đầu : {'image': '<PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=600x353 at 0x7E077BAB74D0', 'label': '1', 'text': 'Andal Amp

In [4]:
# ============================================================
# CELL 4: Dataset class (text-only BERT)
# ============================================================
class MiRAGeBERTDataset(Dataset):
    def __init__(self, hf_dataset, tokenizer, max_len=MAX_LEN):
        self.data      = hf_dataset
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item  = self.data[idx]
        text  = (item.get("caption") or item.get("text") or
                 item.get("title")   or item.get("content") or "")
        enc   = self.tokenizer(
            str(text),
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        label = int(item.get("label", item.get("labels", 0)))
        return (
            enc["input_ids"].squeeze(0),
            enc["attention_mask"].squeeze(0),
            enc["token_type_ids"].squeeze(0),
            label
        )

In [5]:
# ============================================================
# CELL 5: Model BERT + Classifier
# ============================================================
class BERTFakeNewsClassifier(nn.Module):
    def __init__(self, num_classes=2, dropout=0.3):
        super().__init__()
        self.bert = BertModel.from_pretrained("bert-base-uncased")
        hidden    = self.bert.config.hidden_size  # 768
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes)
        )

    def forward(self, input_ids, attention_mask, token_type_ids):
        out     = self.bert(input_ids=input_ids,
                            attention_mask=attention_mask,
                            token_type_ids=token_type_ids)
        # dùng [CLS] token
        cls_out = out.last_hidden_state[:, 0, :]
        return self.classifier(cls_out)

In [6]:
# ============================================================
# CELL 6: Khởi tạo tokenizer, model, dataloader
# ============================================================
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

train_ds = MiRAGeBERTDataset(raw["train"],      tokenizer)
val_ds   = MiRAGeBERTDataset(raw["validation"], tokenizer)

test_splits = [k for k in raw.keys() if k not in ("train", "validation")]
print(f"Test splits: {test_splits}")

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f"Train: {len(train_ds)} | Val: {len(val_ds)}")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Test splits: ['test1_nyt_mj', 'test2_bbc_dalle', 'test3_cnn_dalle', 'test4_bbc_sdxl', 'test5_cnn_sdxl']
Train: 10000 | Val: 2500


In [7]:
# ============================================================
# CELL 7: Training loop + lưu checkpoint
# ============================================================
model     = BERTFakeNewsClassifier().to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.5)

CHECKPOINT_PATH = os.path.join(SAVE_DIR, "latest_epoch.pth")

# ---- Resume nếu có checkpoint ----
start_epoch = 1
if os.path.exists(CHECKPOINT_PATH):
    print("Tìm thấy checkpoint, đang resume...")
    ckpt        = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state"])
    optimizer.load_state_dict(ckpt["optimizer_state"])
    start_epoch = ckpt["epoch"] + 1
    print(f"Tiếp tục từ epoch {start_epoch}")

# ---- Hàm evaluate ----
def evaluate(loader, split_name="Val"):
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0
    with torch.no_grad():
        for input_ids, attn_mask, token_type_ids, labels in tqdm(loader, desc=f"Eval [{split_name}]", leave=False):
            logits     = model(input_ids.to(DEVICE),
                               attn_mask.to(DEVICE),
                               token_type_ids.to(DEVICE))
            loss       = criterion(logits, labels.to(DEVICE))
            total_loss += loss.item()
            all_preds.extend(torch.argmax(logits, dim=1).cpu().tolist())
            all_labels.extend(labels.tolist())

    acc = accuracy_score(all_labels, all_preds)
    f1  = f1_score(all_labels, all_preds, average="macro")
    print(f"  [{split_name}] Loss: {total_loss/len(loader):.4f} | Acc: {acc:.4f} | F1: {f1:.4f}")
    return total_loss / len(loader), acc, f1, all_preds, all_labels

# ---- Training ----
history = []

for epoch in range(start_epoch, EPOCHS + 1):
    model.train()
    running_loss          = 0
    train_preds, train_labels = [], []

    loop = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS} [Train]")
    for input_ids, attn_mask, token_type_ids, labels in loop:
        optimizer.zero_grad()
        logits = model(input_ids.to(DEVICE),
                       attn_mask.to(DEVICE),
                       token_type_ids.to(DEVICE))
        loss   = criterion(logits, labels.to(DEVICE))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        running_loss += loss.item()
        train_preds.extend(torch.argmax(logits, dim=1).cpu().tolist())
        train_labels.extend(labels.tolist())
        loop.set_postfix(loss=f"{loss.item():.4f}")

    train_acc = accuracy_score(train_labels, train_preds)
    train_f1  = f1_score(train_labels, train_preds, average="macro")
    print(f"\nEpoch {epoch} Train → Loss: {running_loss/len(train_loader):.4f} | Acc: {train_acc:.4f} | F1: {train_f1:.4f}")

    _, val_acc, val_f1, _, _ = evaluate(val_loader, "Val")
    scheduler.step()

    # ---- Lưu latest checkpoint ----
    torch.save({
        "epoch":           epoch,
        "model_state":     model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "val_acc":         val_acc,
        "val_f1":          val_f1,
    }, CHECKPOINT_PATH)
    print(f"  ✅ Checkpoint lưu: {CHECKPOINT_PATH}")

    history.append({
        "epoch":     epoch,
        "train_acc": train_acc, "train_f1": train_f1,
        "val_acc":   val_acc,   "val_f1":   val_f1,
    })

print("\n=== TRAINING HOÀN TẤT ===")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Epoch 1/5 [Train]: 100%|██████████| 625/625 [07:48<00:00,  1.33it/s, loss=0.3744]



Epoch 1 Train → Loss: 0.3772 | Acc: 0.8386 | F1: 0.8382


...


Epoch 4 Train → Loss: 0.1081 | Acc: 0.9716 | F1: 0.9716


  [Val] Loss: 0.4037 | Acc: 0.9096 | F1: 0.9095
  ✅ Checkpoint lưu: /content/drive/MyDrive/BERT_MiRAGe/latest_epoch.pth


Epoch 5/5 [Train]: 100%|██████████| 625/625 [07:52<00:00,  1.32it/s, loss=0.0724]



Epoch 5 Train → Loss: 0.0696 | Acc: 0.9820 | F1: 0.9820


  [Val] Loss: 0.4578 | Acc: 0.9088 | F1: 0.9088
  ✅ Checkpoint lưu: /content/drive/MyDrive/BERT_MiRAGe/latest_epoch.pth

=== TRAINING HOÀN TẤT ===


In [8]:
# ============================================================
# CELL 8: Đánh giá tất cả test splits + Lưu kết quả
# ============================================================
# Load checkpoint mới nhất
ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(ckpt["model_state"])
model.eval()
print(f"✅ Load epoch {ckpt['epoch']} | Val Acc: {ckpt['val_acc']:.4f} | Val F1: {ckpt['val_f1']:.4f}\n")

timestamp   = datetime.now().strftime("%Y%m%d_%H%M%S")
all_results = {}

for split in test_splits:
    print(f"{'='*50}\nSplit: {split} ({len(raw[split])} mẫu)")
    ds = MiRAGeBERTDataset(raw[split], tokenizer)
    dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    preds_all, labels_all = [], []
    with torch.no_grad():
        for input_ids, attn_mask, token_type_ids, labels in tqdm(dl, desc=split):
            logits = model(input_ids.to(DEVICE),
                           attn_mask.to(DEVICE),
                           token_type_ids.to(DEVICE))
            preds_all.extend(torch.argmax(logits, dim=1).cpu().tolist())
            labels_all.extend(labels.tolist())

    acc    = accuracy_score(labels_all, preds_all)
    f1     = f1_score(labels_all, preds_all, average="macro")
    report = classification_report(labels_all, preds_all,
                                   target_names=["Real", "AI-Generated"])
    print(report)
    print(f"Accuracy: {acc:.4f} | F1 Macro: {f1:.4f}")

    all_results[split] = {
        "accuracy": float(acc),
        "f1_macro": float(f1),
        "report":   classification_report(labels_all, preds_all,
                                          target_names=["Real", "AI-Generated"],
                                          output_dict=True)
    }

# ---- Bảng tóm tắt ----
print(f"\n{'='*55}")
print("TỔNG KẾT TẤT CẢ TEST SPLITS")
print(f"{'='*55}")
print(f"{'Split':<35} | {'Accuracy':>8} | {'F1 Macro':>8}")
print("-" * 57)
for split, res in all_results.items():
    print(f"{split:<35} | {res['accuracy']:>8.4f} | {res['f1_macro']:>8.4f}")

# ---- Lưu model weights ----
weights_path = os.path.join(SAVE_DIR, "bert_final.pth")
torch.save(model.state_dict(), weights_path)
print(f"\n✅ Model weights: {weights_path}")

# ---- Lưu JSON ----
json_data = {
    "timestamp":       timestamp,
    "checkpoint_epoch": int(ckpt["epoch"]),
    "val_acc":         float(ckpt["val_acc"]),
    "val_f1":          float(ckpt["val_f1"]),
    "test_results":    all_results,
    "training_history": history,
    "config": {
        "model":    "bert-base-uncased",
        "dataset":  "anson-huang/mirage-news",
        "epochs":   EPOCHS,
        "batch":    BATCH_SIZE,
        "lr":       LR,
        "max_len":  MAX_LEN,
    }
}
json_path = os.path.join(SAVE_DIR, f"results_{timestamp}.json")
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(json_data, f, indent=2, ensure_ascii=False)
print(f"✅ JSON report  : {json_path}")

# ---- Lưu TXT ----
txt_path = os.path.join(SAVE_DIR, f"results_{timestamp}.txt")
with open(txt_path, "w", encoding="utf-8") as f:
    f.write(f"BERT Fake News | MiRAGeNews | {timestamp}\n")
    f.write("=" * 55 + "\n")
    f.write(f"Epoch trained : {ckpt['epoch']}/{EPOCHS}\n")
    f.write(f"Val Acc       : {ckpt['val_acc']:.4f}\n")
    f.write(f"Val F1 Macro  : {ckpt['val_f1']:.4f}\n\n")
    f.write(f"{'Split':<35} | {'Accuracy':>8} | {'F1 Macro':>8}\n")
    f.write("-" * 57 + "\n")
    for split, res in all_results.items():
        f.write(f"{split:<35} | {res['accuracy']:>8.4f} | {res['f1_macro']:>8.4f}\n")
    f.write("\n\n--- Training History ---\n")
    f.write(f"{'Epoch':>6} | {'Train Acc':>9} | {'Train F1':>8} | {'Val Acc':>7} | {'Val F1':>7}\n")
    f.write("-" * 52 + "\n")
    for h in history:
        f.write(f"{h['epoch']:>6} | {h['train_acc']:>9.4f} | {h['train_f1']:>8.4f} | {h['val_acc']:>7.4f} | {h['val_f1']:>7.4f}\n")
print(f"✅ TXT report   : {txt_path}")

print("\n========== HOÀN TẤT ==========")
print(f"Drive folder: {SAVE_DIR}")

✅ Load epoch 5 | Val Acc: 0.9088 | Val F1: 0.9088

Split: test1_nyt_mj (500 mẫu)


test1_nyt_mj: 100%|██████████| 32/32 [00:08<00:00,  3.87it/s]


              precision    recall  f1-score   support

        Real       0.96      0.92      0.94       250
AI-Generated       0.93      0.96      0.95       250

    accuracy                           0.94       500
   macro avg       0.94      0.94      0.94       500
weighted avg       0.94      0.94      0.94       500

Accuracy: 0.9440 | F1 Macro: 0.9440
Split: test2_bbc_dalle (500 mẫu)


test2_bbc_dalle: 100%|██████████| 32/32 [00:15<00:00,  2.06it/s]


              precision    recall  f1-score   support

        Real       0.97      0.44      0.61       250
AI-Generated       0.64      0.99      0.78       250

    accuracy                           0.72       500
   macro avg       0.81      0.72      0.69       500
weighted avg       0.81      0.72      0.69       500

Accuracy: 0.7160 | F1 Macro: 0.6933
Split: test3_cnn_dalle (500 mẫu)


...

              precision    recall  f1-score   support

        Real       0.93      0.72      0.81       250
AI-Generated       0.77      0.95      0.85       250

    accuracy                           0.83       500
   macro avg       0.85      0.83      0.83       500
weighted avg       0.85      0.83      0.83       500

Accuracy: 0.8340 | F1 Macro: 0.8318
Split: test4_bbc_sdxl (500 mẫu)


test4_bbc_sdxl: 100%|██████████| 32/32 [00:08<00:00,  3.78it/s]


              precision    recall  f1-score   support

        Real       0.98      0.44      0.61       250
AI-Generated       0.64      0.99      0.78       250

    accuracy                           0.72       500
   macro avg       0.81      0.72      0.70       500
weighted avg       0.81      0.72      0.70       500

Accuracy: 0.7180 | F1 Macro: 0.6951
Split: test5_cnn_sdxl (500 mẫu)


test5_cnn_sdxl: 100%|██████████| 32/32 [00:08<00:00,  3.66it/s]


              precision    recall  f1-score   support

        Real       0.94      0.72      0.82       250
AI-Generated       0.77      0.96      0.86       250

    accuracy                           0.84       500
   macro avg       0.86      0.84      0.84       500
weighted avg       0.86      0.84      0.84       500

Accuracy: 0.8380 | F1 Macro: 0.8357

TỔNG KẾT TẤT CẢ TEST SPLITS
Split                               | Accuracy | F1 Macro
---------------------------------------------------------
test1_nyt_mj                        |   0.9440 |   0.9440
test2_bbc_dalle                     |   0.7160 |   0.6933
test3_cnn_dalle                     |   0.8340 |   0.8318
test4_bbc_sdxl                      |   0.7180 |   0.6951
test5_cnn_sdxl                      |   0.8380 |   0.8357

✅ Model weights: /content/drive/MyDrive/BERT_MiRAGe/bert_final.pth
✅ JSON report  : /content/drive/MyDrive/BERT_MiRAGe/results_20260406_051836.json
✅ TXT report   : /content/drive/MyDrive/BERT_MiRAGe/r